# Synthetic Eval Results Viewer

This notebook reads an existing `synthetic_eval.cli` output directory and shows the key diagnostics: metadata, logs, aggregate metrics, rank histograms, coverage, spread-skill, and example panels.

Default target:

```text
/mnt/sciml/a.sadreev/sea_ice_data/synthetic_eval_rank_swath_daily150x32_4tracks
```

Override it by setting `SYNTH_EVAL_OUT` before launching Jupyter, or edit `OUT_DIR` in the first code cell.

In [ ]:
from __future__ import annotations

import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/mnt/sciml/a.sadreev/sea_ice_data"))
OUT_DIR = Path(os.environ.get("SYNTH_EVAL_OUT", DATA_ROOT / "synthetic_eval_rank_swath_daily150x32_4tracks"))

if not OUT_DIR.exists():
    candidates = sorted(DATA_ROOT.glob("synthetic_eval*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        print(f"Configured OUT_DIR does not exist: {OUT_DIR}")
        OUT_DIR = candidates[0]
        print(f"Using latest synthetic_eval directory instead: {OUT_DIR}")
    else:
        raise FileNotFoundError(f"No synthetic_eval output directories found under {DATA_ROOT}")

PLOTS_DIR = OUT_DIR / "plots"
ARRAYS_DIR = OUT_DIR / "arrays"
print("OUT_DIR =", OUT_DIR)
print("PLOTS_DIR =", PLOTS_DIR)
print("ARRAYS_DIR =", ARRAYS_DIR)

In [ ]:
def read_json(path: Path):
    with open(path) as f:
        return json.load(f)


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def read_table(path: Path):
    if not path.exists():
        print(f"missing: {path}")
        return None
    if pd is not None:
        return pd.read_csv(path)
    return read_csv_rows(path)


def show_png(path: Path, width: int = 980):
    if path.exists():
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path), width=width))
    else:
        print(f"missing: {path}")


def show_many(pattern: str, width: int = 980, limit: int | None = None):
    paths = sorted(PLOTS_DIR.glob(pattern))
    if limit is not None:
        paths = paths[:limit]
    if not paths:
        print(f"no plots matched: {pattern}")
    for path in paths:
        show_png(path, width=width)


def log_tail(path: Path, n: int = 60) -> str:
    if not path.exists():
        return f"missing: {path}"
    lines = path.read_text(errors="replace").splitlines()
    return "\n".join(lines[-n:])


print("available top-level files:")
for path in sorted(OUT_DIR.glob("*")):
    kind = "dir" if path.is_dir() else "file"
    print(f"{kind:4s}  {path.name}")

## Run Metadata And Log

In [ ]:
metadata_path = OUT_DIR / "metadata.json"
if metadata_path.exists():
    metadata = read_json(metadata_path)
    display(metadata)
else:
    print("metadata.json is not written yet")

display(Markdown("**run.log tail**"))
print(log_tail(OUT_DIR / "run.log", n=80))

## Aggregate Metrics

Main table grouped by `variable / mask_type / density / noise_level`. For a swath run with several fake density labels, each density label corresponds to a different swath conditioning seed in the current CLI.

In [ ]:
agg = read_table(OUT_DIR / "aggregate_metrics.csv")
if agg is not None:
    display(agg)
    if pd is not None:
        metric_cols = [c for c in ["rmse", "crps", "energy_score", "spread", "skill_rmse", "spread_skill_ratio"] if c in agg.columns]
        display(agg[["variable", "mask_type", "density", "noise_level", *metric_cols]])

## Coverage Reliability

If calibrated, empirical coverage should be close to nominal levels: 0.5, 0.8, 0.9, 0.95.

In [ ]:
if pd is not None and agg is not None:
    coverage_cols = [c for c in agg.columns if c.startswith("coverage_")]
    if coverage_cols:
        coverage = agg[["variable", "mask_type", "density", "noise_level", *coverage_cols]].copy()
        display(coverage)

        fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, variable in zip(axes, sorted(coverage["variable"].unique())):
            sub = coverage[coverage["variable"] == variable]
            for _, row in sub.iterrows():
                nominal = np.array([float(c.split("_")[1]) for c in coverage_cols])
                empirical = row[coverage_cols].astype(float).to_numpy()
                label = f"{row['mask_type']} d={row['density']:g} n={row['noise_level']:g}"
                ax.plot(nominal, empirical, marker="o", label=label)
            ax.plot([0, 1], [0, 1], "k--", linewidth=1)
            ax.set_title(variable)
            ax.set_xlabel("nominal coverage")
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel("empirical coverage")
        axes[-1].legend(fontsize=8, bbox_to_anchor=(1.04, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
    else:
        print("no coverage columns found")
else:
    print("pandas is unavailable or aggregate metrics are missing")

## Rank Histograms From NPZ

This recomputes the plots from `arrays/rank_histograms.npz` and prints simple diagnostics. A flat histogram is the target for marginal ensemble calibration.

In [ ]:
rank_path = ARRAYS_DIR / "rank_histograms.npz"
if rank_path.exists():
    rank_data = np.load(rank_path)
    rows = []
    n = len(rank_data.files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.3 * nrows), squeeze=False)
    for ax, key in zip(axes.ravel(), sorted(rank_data.files)):
        counts = rank_data[key].astype(np.float64)
        total = counts.sum()
        probs = counts / total if total else counts
        expected = 1.0 / len(counts)
        bins = np.arange(len(counts))
        ax.bar(bins, probs, color="#4c78a8")
        ax.axhline(expected, color="black", linestyle="--", linewidth=1)
        ax.set_title(key)
        ax.set_xlabel("truth rank among ensemble members")
        ax.set_ylabel("frequency")
        ax.grid(True, axis="y", alpha=0.25)

        left_edge = probs[0]
        right_edge = probs[-1]
        edge_ratio = (left_edge + right_edge) / (2 * expected)
        center = probs[len(probs) // 2]
        chi2_like = float(((counts - total * expected) ** 2 / max(total * expected, 1.0)).sum()) if total else float("nan")
        rows.append({
            "key": key,
            "bins": len(counts),
            "total_ranks": int(total),
            "expected_per_bin": total * expected,
            "left_edge_freq": left_edge,
            "right_edge_freq": right_edge,
            "edge_ratio_vs_flat": edge_ratio,
            "center_freq": center,
            "chi2_like_not_independent": chi2_like,
        })
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        display(rows)
else:
    print(f"rank histogram file is missing: {rank_path}")

## Saved Plot Files

These are the exact plots written by `synthetic_eval.cli`.

In [ ]:
display(Markdown("### Example panels"))
show_many("example_*.png", width=1050)

display(Markdown("### Rank histograms"))
show_many("rank_hist_*.png", width=900)

display(Markdown("### Coverage plots"))
show_many("coverage_*.png", width=900)

display(Markdown("### Spread-skill plots"))
show_many("spread_skill_*.png", width=900)

display(Markdown("### Density curves"))
show_many("rmse_vs_density.png", width=900)
show_many("crps_vs_density.png", width=900)

## Per-Case Distributions

In [ ]:
per_case = read_table(OUT_DIR / "per_case_metrics.csv")
if pd is not None and per_case is not None:
    display(per_case.head())
    metrics = [c for c in ["rmse", "crps", "spread_skill_ratio", "observed_fraction"] if c in per_case.columns]
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(9, 4))
        for variable, sub in per_case.groupby("variable"):
            ax.hist(sub[metric].astype(float), bins=30, alpha=0.55, label=variable)
        ax.set_title(metric)
        ax.set_xlabel(metric)
        ax.set_ylabel("case rows")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.show()
else:
    print("per_case_metrics.csv is missing or pandas is unavailable")

## Saved Tensor Archives

If the run used `--save-tensors`, this section lists `.npz` files and previews one archive. Expected keys: `ensemble`, `truth`, `mask`, `observed`.

In [ ]:
SAMPLES_DIR = OUT_DIR / "samples"
tensor_paths = sorted(SAMPLES_DIR.glob("**/*.npz"))
print("SAMPLES_DIR =", SAMPLES_DIR)
print("n tensor archives =", len(tensor_paths))
for path in tensor_paths[:20]:
    print(path.relative_to(OUT_DIR))
if len(tensor_paths) > 20:
    print(f"... {len(tensor_paths) - 20} more")

if tensor_paths:
    sample_path = tensor_paths[0]
    payload = np.load(sample_path, allow_pickle=False)
    print("preview archive =", sample_path)
    for key in payload.files:
        arr = payload[key]
        print(f"{key:16s} shape={arr.shape} dtype={arr.dtype}")

    ensemble = payload["ensemble"]
    truth = payload["truth"]
    mask = payload["mask"]
    observed = payload["observed"]
    mean = ensemble.mean(axis=0)

    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    axes = axes.ravel()
    panels = [
        (truth[0], "truth concentration", "viridis"),
        (observed[0], "observed concentration", "viridis"),
        (mask, "track mask", "gray"),
        (mean[0], "ensemble mean concentration", "viridis"),
        (ensemble[0, 0], "sample 0 concentration", "viridis"),
        (ensemble[min(1, ensemble.shape[0] - 1), 0], "sample 1 concentration", "viridis"),
        (ensemble[min(2, ensemble.shape[0] - 1), 0], "sample 2 concentration", "viridis"),
        (mean[0] - truth[0], "mean - truth concentration", "coolwarm"),
    ]
    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No saved tensors found. Rerun synthetic_eval.cli with --save-tensors.")

## Notes For Interpretation

- Rank histograms are marginal calibration diagnostics over selected grid points, not independent-pixel p-value tests.
- A U-shaped rank histogram usually means the ensemble is underdispersed.
- A center-heavy rank histogram usually means the ensemble is overdispersed.
- A left/right tilt indicates bias.
- Current `synthetic_eval.cli` uses valid pixels for rank histograms; it does not yet exclude observed swath pixels from the rank calculation.
- For swath runs with multiple density labels, the labels are used to create multiple conditioning seeds in the current setup; swath coverage itself is produced by the track-mask generator.